# Conector Otimizado Oracle para Spark

Módulo corporativo para conexão com o **Oracle Database** no Apache Spark.

### Recursos de Destaque:
- **Preservação Estrita do Bootstrap JVM:** Mantém intacta a lógica de `DriverManager` e `URLClassLoader` com `ojdbc8.jar`.
- **Alta Disponibilidade:** Suporte a Failover e Load Balancing (Host 1 + Host 2).
- **Operações Transacionais:** `command` com commit/rollback seguro, `write` em lotes, `clear` (DELETE em lotes ou TRUNCATE) e `reload`.
- **Linguagem Natural Spark:** `sql`, `table`, `command`, `write`, `clear`, `reload`, `limit`.


## 1. Conector Oracle (`ConectorOracleSpark`)


In [ ]:
%%spark

class ConectorOracleSpark:
    """
    Conector corporativo para Oracle Database no Apache Spark.
    """
    DRIVER_PADRAO = "oracle.jdbc.OracleDriver"

    def __init__(
        self,
        spark: SparkSession,
        env: Optional[Dict[str, str]] = None,
    ) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)

        def extrair_obrigatorio(chave: str) -> str:
            val = self.env.get(chave)
            if val is None or not str(val).strip():
                raise ValueError(f"Variável obrigatória do Oracle não informada: '{chave}'")
            return str(val).strip()

        def extrair_opcional(chave: str) -> Optional[str]:
            val = self.env.get(chave)
            return str(val).strip() if val and str(val).strip() else None

        self.usuario = extrair_obrigatorio("VDP_ORACLE_USER")
        self.senha = extrair_obrigatorio("VDP_ORACLE_PASSWORD")
        self.host_1 = extrair_obrigatorio("VDP_ORACLE_HOST_1")
        self.host_2 = extrair_opcional("VDP_ORACLE_HOST_2")
        self.porta = extrair_obrigatorio("VDP_ORACLE_PORTA")
        self.service_name = extrair_opcional("VDP_ORACLE_SERVICE_NAME") or extrair_obrigatorio("VDP_ORACLE_SERVICE")
        self.schema = (extrair_opcional("VDP_ORACLE_SCHEMA") or self.usuario).upper()
        self.driver = extrair_opcional("VDP_ORACLE_DRIVER") or self.DRIVER_PADRAO
        self.jar_path = extrair_opcional("VDP_ORACLE_JAR") or "/dados/shared/bin/ojdbc8.jar"

        if self.driver != self.DRIVER_PADRAO:
            raise ValueError(f"Driver Oracle inválido: {self.driver}")

        if self.host_2:
            self.url = (
                "jdbc:oracle:thin:@(DESCRIPTION="
                "(LOAD_BALANCE=OFF)"
                "(FAILOVER=ON)"
                "(CONNECT_TIMEOUT=10)"
                "(TRANSPORT_CONNECT_TIMEOUT=3)"
                "(RETRY_COUNT=3)"
                "(ADDRESS_LIST="
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_1})(PORT={self.porta}))"
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_2})(PORT={self.porta}))"
                ")"
                f"(CONNECT_DATA=(SERVICE_NAME={self.service_name}))"
                ")"
            )
        else:
            self.url = f"jdbc:oracle:thin:@//{self.host_1}:{self.porta}/{self.service_name}"

    def sql(
        self,
        query: str,
        fetchsize: Optional[int] = 5_000,
        query_timeout: Optional[int] = None,
        partition_column: Optional[str] = None,
        lower_bound: Any = None,
        upper_bound: Any = None,
        num_partitions: Optional[int] = None,
        show: bool = False,
        truncate: bool = True,
        n: int = 20,
    ) -> DataFrame:
        """
        Executa uma consulta SQL no Oracle e retorna um DataFrame Spark lazy.
        """
        instrucao_sql = (query or "").strip()
        if not instrucao_sql:
            raise ValueError("Instrução SQL não pode ser vazia.")
        if instrucao_sql.endswith(";"):
            instrucao_sql = instrucao_sql[:-1].strip()

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.usuario)
            .option("password", self.senha)
            .option("dbtable", f"({instrucao_sql}) T")
        )

        if fetchsize is not None:
            reader = reader.option("fetchsize", int(fetchsize))
        if query_timeout is not None:
            reader = reader.option("queryTimeout", int(query_timeout))

        if any(v is not None for v in [partition_column, lower_bound, upper_bound, num_partitions]):
            if any(v is None for v in [partition_column, lower_bound, upper_bound, num_partitions]):
                raise ValueError("Para particionamento Oracle, informe partition_column, lower_bound, upper_bound e num_partitions.")
            reader = (
                reader
                .option("partitionColumn", str(partition_column).strip().upper())
                .option("lowerBound", str(lower_bound).strip())
                .option("upperBound", str(upper_bound).strip())
                .option("numPartitions", int(num_partitions))
            )

        df = reader.load()
        if show:
            df.show(n=int(n), truncate=truncate)
        return df

    def table(
        self,
        table_name: str,
        schema: Optional[str] = None,
        columns: Optional[List[str]] = None,
        partition_column: Optional[str] = None,
        lower_bound: Any = None,
        upper_bound: Any = None,
        fetchsize: Optional[int] = 5_000,
        num_partitions: Optional[int] = None,
        show: bool = False,
        truncate: bool = True,
        n: int = 20,
    ) -> DataFrame:
        final_owner = (schema or self.schema or "").strip().upper()
        tabela = (table_name or "").strip().upper()
        if not tabela or not final_owner:
            raise ValueError("Nome da tabela e schema/owner devem ser informados.")

        cols_sql = ", ".join(columns) if columns else "*"
        query_sql = f"SELECT {cols_sql} FROM {final_owner}.{tabela}"
        return self.sql(
            query=query_sql,
            fetchsize=fetchsize,
            partition_column=partition_column,
            lower_bound=lower_bound,
            upper_bound=upper_bound,
            num_partitions=num_partitions,
            show=show,
            truncate=truncate,
            n=n,
        )

    def command(self, sql: str) -> None:
        """
        Executa um comando DDL ou DML direto no Oracle via JDBC nativo (JVM / URLClassLoader).
        """
        instrucao = (sql or "").strip()
        if not instrucao:
            raise ValueError("Instrução SQL não pode ser vazia.")
        if instrucao.endswith(";"):
            instrucao = instrucao[:-1].strip()

        jvm = self.spark._jvm
        conn = None
        stmt = None

        def criar_conexao_driver_manager():
            try:
                jvm.java.lang.Class.forName(self.driver)
            except Exception:
                context_loader = jvm.java.lang.Thread.currentThread().getContextClassLoader()
                jvm.java.lang.Class.forName(self.driver, True, context_loader)
            return jvm.java.sql.DriverManager.getConnection(self.url, self.usuario, self.senha)

        def criar_conexao_url_classloader():
            gateway = self.spark.sparkContext._gateway
            jar_file = jvm.java.io.File(self.jar_path)
            if not jar_file.exists():
                raise ValueError(f"Jar Oracle não encontrado em: '{self.jar_path}'.")
            jar_url = jar_file.toURI().toURL()
            urls = gateway.new_array(jvm.java.net.URL, 1)
            urls[0] = jar_url
            parent_loader = jvm.java.lang.Thread.currentThread().getContextClassLoader()
            loader = jvm.java.net.URLClassLoader(urls, parent_loader)
            driver_class = jvm.java.lang.Class.forName(self.driver, True, loader)
            driver_inst = driver_class.newInstance()
            props = jvm.java.util.Properties()
            props.setProperty("user", self.usuario)
            props.setProperty("password", self.senha)
            return driver_inst.connect(self.url, props)

        try:
            try:
                conn = criar_conexao_driver_manager()
            except Exception:
                conn = criar_conexao_url_classloader()

            conn.setAutoCommit(False)
            stmt = conn.createStatement()
            stmt.execute(instrucao)
            conn.commit()
        except Exception:
            if conn is not None:
                try:
                    conn.rollback()
                except Exception:
                    pass
            raise
        finally:
            if stmt is not None:
                try:
                    stmt.close()
                except Exception:
                    pass
            if conn is not None:
                try:
                    conn.close()
                except Exception:
                    pass

    def write(
        self,
        df: DataFrame,
        table_name: str,
        schema: Optional[str] = None,
        batch_size: int = 5000,
        num_partitions: int = 1,
    ) -> None:
        """
        Grava um DataFrame Spark em uma tabela Oracle via JDBC append em lotes.
        """
        if df is None:
            raise ValueError("DataFrame não pode ser nulo.")
        tabela = (table_name or "").strip().upper()
        final_owner = (schema or self.schema or "").strip().upper()
        if not tabela or not final_owner:
            raise ValueError("Tabela e schema/owner devem ser informados.")

        full_table = f"{final_owner}.{tabela}"
        writer_df = df.coalesce(int(num_partitions)) if int(num_partitions) > 0 else df
        (
            writer_df.write
            .format("jdbc")
            .mode("append")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.usuario)
            .option("password", self.senha)
            .option("dbtable", full_table)
            .option("batchsize", int(batch_size))
            .save()
        )

    def clear(
        self,
        table_name: str,
        mode: str = "truncate",
        batch_size: Optional[int] = None,
        schema: Optional[str] = None,
    ) -> None:
        """
        Limpa os dados de uma tabela Oracle usando TRUNCATE ou DELETE em lotes.
        """
        tabela = (table_name or "").strip().upper()
        final_owner = (schema or self.schema or "").strip().upper()
        if not tabela or not final_owner:
            raise ValueError("Tabela e schema/owner devem ser informados.")

        modo = (mode or "truncate").lower()
        if modo == "delete" and batch_size is None:
            raise ValueError("batch_size is required when mode='delete'")

        lote = batch_size or 5000
        full_table = f"{final_owner}.{tabela}"

        if modo == "truncate":
            self.command(f"TRUNCATE TABLE {full_table}")
            return

        if modo == "delete":
            lote_num = int(lote)
            while True:
                df_count = self.sql(f"SELECT COUNT(1) AS QTD FROM {full_table}")
                qtd_restante = int(df_count.collect()[0]["QTD"] or 0)
                if qtd_restante == 0:
                    break
                self.command(f"DELETE FROM {full_table} WHERE ROWNUM <= {lote_num}")
            return

        raise ValueError(f"Modo de limpeza inválido: '{modo}'. Utilize 'truncate' ou 'delete'.")

    def reload(
        self,
        df: DataFrame,
        table_name: str,
        schema: Optional[str] = None,
        clear_mode: str = "truncate",
        delete_batch_size: Optional[int] = None,
        batch_size: int = 5000,
        num_partitions: int = 1,
    ) -> None:
        """
        Operação composta: clear + write (substituição / recarga de tabela).
        """
        self.clear(
            table_name=table_name,
            mode=clear_mode,
            batch_size=delete_batch_size,
            schema=schema,
        )
        self.write(
            df=df,
            table_name=table_name,
            schema=schema,
            batch_size=batch_size,
            num_partitions=num_partitions,
        )

    def limit(self, table_name: str, schema: Optional[str] = None, limit: int = 100) -> DataFrame:
        """
        Obtém uma amostra de linhas do Oracle para análise exploratória.
        """
        final_owner = (schema or self.schema or "").strip().upper()
        tabela = (table_name or "").strip().upper()
        query_sql = f"SELECT * FROM {final_owner}.{tabela} WHERE ROWNUM <= {int(limit)}"
        return self.sql(query_sql, fetchsize=min(limit, 5000))


def criar_conector_oracle_spark(env: Optional[Dict[str, str]] = None) -> ConectorOracleSpark:
    return ConectorOracleSpark(spark=spark, env=env)

print("Módulo gerenciador_oracle_spark carregado com sucesso (ConectorOracleSpark).")
